# 04-ownership: 객체 수명과 소유권

목표는 C++에서 값 객체, RAII, smart pointer, `std::optional`을 어떻게 쓰는지 기본 흐름을 익히는 것입니다.

## 값 객체와 캡슐화

작은 타입은 값으로 만들고, 상태 변경 규칙은 멤버 함수 안에 둡니다.

In [ ]:
#include <iostream>
#include <string>
#include <utility>

{
    class Account {
    public:
        Account(std::string owner, int balance) : owner_(std::move(owner)), balance_(balance) {}

        void deposit(int amount) {
            if (amount > 0) balance_ += amount;
        }

        bool withdraw(int amount) {
            if (amount <= 0 || amount > balance_) return false;
            balance_ -= amount;
            return true;
        }

        const std::string& owner() const { return owner_; }
        int balance() const { return balance_; }

    private:
        std::string owner_;
        int balance_;
    };

    Account account{"kim", 10000};
    account.deposit(5000);
    account.withdraw(3000);

    std::cout << account.owner() << ": " << account.balance() << "\n";
}

## RAII와 `std::unique_ptr`

RAII는 자원의 획득과 해제를 객체 수명에 묶는 방식입니다. 직접 `new`/`delete`를 쓰기보다 소유권을 표현하는 타입을 사용합니다.

In [ ]:
#include <iostream>
#include <memory>
#include <string>
#include <utility>

{
    struct Profile {
        std::string name;
        int level;
    };

    auto profile = std::make_unique<Profile>(Profile{"kim", 3});
    std::cout << profile->name << " level " << profile->level << "\n";

    auto moved = std::move(profile);
    std::cout << "profile moved: " << std::boolalpha << (profile == nullptr) << "\n";
    std::cout << moved->name << "\n";
}

## `std::shared_ptr`

여러 곳이 같은 객체를 공유해야 할 때만 `shared_ptr`를 사용합니다. 기본값은 값 객체 또는 `unique_ptr`가 더 단순합니다.

In [ ]:
#include <iostream>
#include <memory>
#include <string>

{
    auto shared = std::make_shared<std::string>("shared data");
    auto alias = shared;

    std::cout << *shared << "\n";
    std::cout << "use_count: " << shared.use_count() << "\n";
    alias.reset();
    std::cout << "after reset: " << shared.use_count() << "\n";
}

## `std::optional`

값이 없을 수 있음을 반환 타입으로 표현할 때 `std::optional<T>`를 씁니다. 실패를 숫자 `-1` 같은 임시 약속으로 숨기지 않습니다.

In [ ]:
#include <iostream>
#include <optional>
#include <string>

{
    auto parse_positive = [](const std::string& text) -> std::optional<int> {
        try {
            int value = std::stoi(text);
            if (value > 0) return value;
        } catch (...) {
        }
        return std::nullopt;
    };

    for (const auto& text : {std::string{"42"}, std::string{"-1"}, std::string{"cpp"}}) {
        auto value = parse_positive(text);
        if (value) std::cout << text << " -> " << *value << "\n";
        else std::cout << text << " -> invalid\n";
    }
}

## 실습

`Account`에 `transfer_to(Account& other, int amount)`를 추가해 보세요. 출금이 성공했을 때만 상대 계좌에 입금해야 합니다.

## 체크포인트

- 값 객체는 복사와 수명이 단순해서 기본 선택지로 좋다.
- 직접 `new`/`delete`를 쓰지 말고 RAII 타입을 우선한다.
- 단독 소유는 `unique_ptr`, 공유 소유는 `shared_ptr`로 표현한다.
- 값이 없을 수 있으면 `std::optional<T>`로 타입에 드러낸다.